# Sparse-scan SEM reconstruction — U-Net training on Colab

Runtime → Change runtime type → **GPU (A100 or L4)**, high-RAM if available.

Storage layout:
- **Google Drive** (`DRIVE_DIR`) is the persistent store: the 10 dataset tars (~13.5 GB, downloaded once), `splits.csv`, the frozen eval sets and all checkpoints.
- **Local `/content` disk** holds the extracted images for the session. Reading ~21k JPEGs directly from the Drive FUSE mount is very slow, so each session copies the tars from Drive and extracts them locally (a few minutes). The trade-off is that step repeats every session and needs ~14 GB of Drive space; if Drive space is short, set `TARS_ON_DRIVE = False` to download straight from B2SHARE each session instead (slower, depends on B2SHARE availability).

Training resumes automatically from `DRIVE_DIR/checkpoints/<run>/last.pt`, so a disconnected session can simply be re-run top to bottom.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os
print("CPUs:", os.cpu_count())

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

REPO_URL = "https://github.com/CameronGordonn/sparse-scan-SEM-reconstruction.git"  # private repo: prefix with https://<token>@github.com/... or make public
DRIVE_DIR = "/content/drive/MyDrive/sem-recon"
TARS_ON_DRIVE = True
LOCAL_DATA = "/content/nffa"
os.makedirs(DRIVE_DIR, exist_ok=True)

In [ ]:
%cd /content
!test -d repo || git clone -q $REPO_URL repo
%cd /content/repo
!git pull -q
!pip install -q -e .

## Data: tars on Drive (once), extracted to local disk (every session)

In [ ]:
import subprocess, tarfile, urllib.request
from semrecon.data import B2SHARE_FILES, CATEGORIES

tar_dir = f"{DRIVE_DIR}/nffa_tars" if TARS_ON_DRIVE else "/content/nffa_tars"
os.makedirs(tar_dir, exist_ok=True)
os.makedirs(LOCAL_DATA, exist_ok=True)
for cat in CATEGORIES:
    if os.path.isdir(f"{LOCAL_DATA}/{cat}") and os.listdir(f"{LOCAL_DATA}/{cat}"):
        continue
    tar = f"{tar_dir}/{cat}.tar"
    if not os.path.exists(tar):
        print("downloading", cat)
        subprocess.run(["curl", "-sSL", "--retry", "5", "-o", tar + ".part", B2SHARE_FILES.format(name=cat)], check=True)
        os.rename(tar + ".part", tar)
    print("extracting", cat)
    local_tar = f"/content/{cat}.tar"
    subprocess.run(["cp", tar, local_tar], check=True)   # sequential read from Drive, then fast local extract
    with tarfile.open(local_tar) as tf:
        tf.extractall(LOCAL_DATA, filter="data")
    os.remove(local_tar)
print({c: len(os.listdir(f"{LOCAL_DATA}/{c}")) for c in CATEGORIES})

In [ ]:
# splits + frozen eval sets live on Drive so every session (and the evaluator) uses the same ones
SPLITS = f"{DRIVE_DIR}/splits.csv"
EVAL_TEST = f"{DRIVE_DIR}/eval_test.npz"
EVAL_VAL = f"{DRIVE_DIR}/eval_val.npz"
if not os.path.exists(EVAL_TEST):
    !python scripts/prepare_data.py --skip-download --root $LOCAL_DATA --splits $SPLITS --eval-out $EVAL_TEST --val-out $EVAL_VAL
else:
    print("using existing splits/eval sets on Drive")

## Train (single model across all patterns, fractions and doses)

In [ ]:
RUN = "unet"
WORKERS = max(2, os.cpu_count() - 1)
!python scripts/train_unet.py --config configs/unet.yaml \
    --out-dir $DRIVE_DIR/checkpoints/$RUN \
    --set data.root=$LOCAL_DATA --set data.splits=$SPLITS --set data.val_set=$EVAL_VAL \
    --set data.num_workers=$WORKERS

## Ablation: pattern-specific model (uniform-only training)
Shows the generalization gap when a model trained on uniform random pixels is evaluated on scan-feasible patterns.

In [ ]:
!python scripts/train_unet.py --config configs/unet.yaml \
    --out-dir $DRIVE_DIR/checkpoints/unet_uniform_only \
    --set data.root=$LOCAL_DATA --set data.splits=$SPLITS --set data.val_set=$EVAL_VAL \
    --set data.num_workers=$WORKERS --set "sampling.patterns=[uniform]" --set train.steps=30000

In [ ]:
import json, matplotlib.pyplot as plt
h = json.load(open(f"{DRIVE_DIR}/checkpoints/{RUN}/history.json"))
steps = [r["step"] for r in h]
for k in [k for k in h[0] if k.startswith("psnr")]:
    plt.plot(steps, [r[k] for r in h], label=k)
plt.xlabel("step"); plt.ylabel("val PSNR (dB)"); plt.legend(); plt.show()

## Evaluate on the frozen test set
`scripts/evaluate.py` reads `data/eval_test.npz` / `data/eval_val.npz` (see `configs/eval.yaml`), so link them to the Drive copies. U-Net rows are appended to a CSV on Drive; merge with the classical-baseline CSV for plotting.

In [ ]:
os.makedirs("data", exist_ok=True)
for src, dst in [(EVAL_TEST, "data/eval_test.npz"), (EVAL_VAL, "data/eval_val.npz"), (SPLITS, "data/splits.csv")]:
    if not os.path.exists(dst):
        os.symlink(src, dst)
os.makedirs(f"{DRIVE_DIR}/results", exist_ok=True)
for run in ["unet", "unet_uniform_only"]:
    ckpt = f"{DRIVE_DIR}/checkpoints/{run}/best.pt"
    if os.path.exists(ckpt):
        !python scripts/evaluate.py --stage eval --methods unet --unet-ckpt $ckpt --method-name {run} --device cuda --out $DRIVE_DIR/results/metrics_{run}.csv

## Diffusion model (day 5)
Conditional DDPM with inputs [x_t, y, mask], trained on 128² patches across all patterns, fractions and doses (`configs/diffusion.yaml`, ~7.9M params).
When training ends, the best EMA checkpoint is scored on val with each measurement-consistency mode (`none` / `repaint` / `hard`); the winner is stored in `best.pt` as the default sampler and the scores go to `consistency.json`.
Training resumes from `last.pt`, so rerun the cell after a Colab disconnect.

In [ ]:
DIFF_RUN = "diffusion"
!python scripts/train_diffusion.py --config configs/diffusion.yaml \
    --out-dir $DRIVE_DIR/checkpoints/$DIFF_RUN \
    --set data.root=$LOCAL_DATA --set data.splits=$SPLITS --set data.val_set=$EVAL_VAL \
    --set data.num_workers=$WORKERS

In [ ]:
print(open(f"{DRIVE_DIR}/checkpoints/{DIFF_RUN}/consistency.json").read())
# Evaluate on the frozen 512² test crops (fully convolutional inference, 4-sample ensemble mean).
# Cost scales with n_images x 30 cases x samples x steps; 50 images at 100 steps is ~1-2 h on an A100.
DIFF_EVAL_IMAGES = 50
ckpt = f"{DRIVE_DIR}/checkpoints/{DIFF_RUN}/best.pt"
!python scripts/evaluate.py --stage eval --methods diffusion --diffusion-ckpt $ckpt --diffusion-samples 4 \
    --n-images $DIFF_EVAL_IMAGES --device cuda --out $DRIVE_DIR/results/metrics_{DIFF_RUN}.csv